# Prompt Generation Debug Notebook
## Inspect Components Step-by-Step

This notebook breaks down the prompt generation process to identify where dataset description and feature information may be missing.

## 1. Load and Display Dataset Info

In [ ]:
import pickle
import pandas as pd
import numpy as np
from pathlib import Path

# Load dataset_info from pickle file
DATASET_INFO_PATH = Path('datasets_prep/data/saudi_dataset/dataset_info')

print(f"Loading from: {DATASET_INFO_PATH}")
print(f"File exists: {DATASET_INFO_PATH.exists()}")

with open(DATASET_INFO_PATH, 'rb') as f:
    DATASET_INFO = pickle.load(f)

print(f"\nDataset Info loaded successfully!")
print(f"Keys in DATASET_INFO: {list(DATASET_INFO.keys())}")

## 2. Extract and Display Dataset Description

In [ ]:
# Extract the three main descriptions
dataset_description = DATASET_INFO.get("dataset_description", "")
target_description = DATASET_INFO.get("target_description", "")
task_description = DATASET_INFO.get("task_description", "")

print("=" * 80)
print("COMPONENT 1: DATASET DESCRIPTION")
print("=" * 80)
print(f"Length: {len(dataset_description)} characters")
print(f"\nContent:\n{dataset_description}")

print("\n" + "=" * 80)
print("COMPONENT 2: TARGET DESCRIPTION")
print("=" * 80)
print(f"Length: {len(target_description)} characters")
print(f"\nContent:\n{target_description}")

print("\n" + "=" * 80)
print("COMPONENT 3: TASK DESCRIPTION")
print("=" * 80)
print(f"Length: {len(task_description)} characters")
print(f"\nContent:\n{task_description}")

## 3. Extract Feature Description DataFrame

In [ ]:
# Extract feature description
feature_df = DATASET_INFO.get("feature_description")

print("=" * 80)
print("COMPONENT 4: FEATURE DESCRIPTIONS DataFrame")
print("=" * 80)
print(f"Shape: {feature_df.shape}")
print(f"Columns: {list(feature_df.columns)}")
print(f"\nFirst 5 rows:")
print(feature_df.head())

## 4. Calculate and Display Feature Averages

In [ ]:
print("=" * 80)
print("COMPONENT 5: FEATURE AVERAGES")
print("=" * 80)
print(f"\nAll features with averages:")
for idx, row in feature_df.iterrows():
    feature_name = row['feature_name']
    feature_avg = row['feature_average']
    feature_desc = row['feature_desc']
    print(f"  {feature_name}: avg = {feature_avg:.4f}")

## 5. Extract Categorical Meanings

In [ ]:
print("=" * 80)
print("COMPONENT 6: CATEGORICAL MEANINGS (from descriptions)")
print("=" * 80)
print(f"\nAll features with their categorical meanings:")
for idx, row in feature_df.iterrows():
    feature_name = row['feature_name']
    feature_desc = row['feature_desc']
    print(f"\n{feature_name}:")
    print(f"  {feature_desc}")

## 6. Load Adverse Data Examples

In [ ]:
# Load adverse data
df = pd.read_csv('datasets_prep/data/saudi_dataset/saudi_adverse.csv')

print("=" * 80)
print("COMPONENT 7: ADVERSE DATA")
print("=" * 80)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst row example:")
print(df.iloc[0])

## 7. Create Instance Description with Feature Info

In [ ]:
# Take first row
row = df.iloc[0].drop(['Unnamed: 0', 'predicted_class', 'prediction_score', 'actual_target'])
prediction = int(df.iloc[0]['predicted_class'])

print("=" * 80)
print("COMPONENT 8: INSTANCE DESCRIPTION WITH AVERAGES")
print("=" * 80)

feature_lines = []
for col in row.index:
    value = row[col]
    # Find feature description
    feature_info = feature_df[feature_df['feature_name'] == col]
    if not feature_info.empty:
        desc = feature_info.iloc[0]['feature_desc']
        avg = feature_info.iloc[0]['feature_average']
        # Handle numeric values
        try:
            value_str = f"{float(value):.2f}"
            avg_str = f"{float(avg):.2f}"
            feature_lines.append(f"- {col} = {value_str} (avg: {avg_str}) - {desc}")
        except (ValueError, TypeError):
            feature_lines.append(f"- {col} = {value} - {desc}")
    else:
        feature_lines.append(f"- {col} = {value}")

instance_desc = f"""The model is making a prediction for a job applicant.

Feature values:
{chr(10).join(feature_lines)}
"""

print(instance_desc)

## 8. Assemble Full Prompt - Check Each Component

In [ ]:
print("=" * 80)
print("CHECKING FINAL PROMPT ASSEMBLY")
print("=" * 80)

# Build the full prompt
DATASET_EXPLANATION = """1. DATASET INFORMATION"""

FULL_DATASET_DESC = f"""{dataset_description}

Target: {target_description}

ML Task: {task_description}"""

APPLICANT_INFORMATION = """3. APPLICANT PROFILE"""

full_prompt = f"""PREAMBLE:
A machine learning model predicted that a job applicant will LEAVE their position.

{DATASET_EXPLANATION}
{FULL_DATASET_DESC}

{APPLICANT_INFORMATION}
{instance_desc}
"""

print(f"\nFull prompt length: {len(full_prompt)} characters")
print(f"\nContains 'DATASET INFORMATION': {'DATASET INFORMATION' in full_prompt}")
print(f"Contains dataset description: {'Saudi private sector' in full_prompt}")
print(f"Contains feature averages: {'(avg:' in full_prompt}")
print(f"Contains categorical meanings: {any(col in full_prompt for col in feature_df['feature_name'].tolist())}")

print(f"\n" + "=" * 80)
print("FINAL PROMPT PREVIEW:")
print("=" * 80)
print(full_prompt[:1500] + "\n... [truncated] ...")

## 9. Verify All Components Are Present

In [ ]:
print("=" * 80)
print("COMPONENT VERIFICATION CHECKLIST")
print("=" * 80)

checks = {
    "✓ Dataset description loaded": len(dataset_description) > 0,
    "✓ Target description loaded": len(target_description) > 0,
    "✓ Task description loaded": len(task_description) > 0,
    "✓ Feature DataFrame loaded": feature_df is not None and len(feature_df) > 0,
    "✓ Feature averages available": all(pd.notna(feature_df['feature_average'])),
    "✓ Feature descriptions available": all(pd.notna(feature_df['feature_desc'])),
    "✓ Adverse data loaded": len(df) > 0,
    "✓ All features in adverse data have info": all(col in feature_df['feature_name'].values for col in row.index),
    "✓ Instance description created": len(instance_desc) > 0,
    "✓ Prompt contains dataset section": 'DATASET INFORMATION' in full_prompt,
    "✓ Prompt contains feature averages": '(avg:' in full_prompt,
}

for check, result in checks.items():
    status = "PASS" if result else "FAIL"
    print(f"{check} ... [{status}]")

all_pass = all(checks.values())
print(f"\n{'🎉 All checks passed!' if all_pass else '⚠️ Some checks failed - see above'}")

## 10. Save the Debug Output

In [ ]:
# Save the full prompt for inspection
with open('debug_prompt_output.txt', 'w') as f:
    f.write(full_prompt)

print(f"Full prompt saved to: debug_prompt_output.txt")
print(f"Prompt size: {len(full_prompt)} characters")